In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

BASE = Path.cwd().parent
DATA = BASE / "Datasets" / "processed" / "dataset.csv"

df = pd.read_csv(DATA).fillna({"title": "", "abstract": "", "ref_dois": ""})
label_cols = [c for c in df.columns if c.startswith("lbl_")]

print(f"papers: {len(df)}")
print(f"missing abstracts: {(df.abstract.str.len() == 0).sum()}")
print(f"years: {df.year.min()} to {df.year.max()}")
print(f"\nreferences per paper: median {df.n_refs.median():.0f}, "
      f"mean {df.n_refs.mean():.1f}, max {df.n_refs.max()}")

papers: 8192
missing abstracts: 40
years: 2020 to 2026

references per paper: median 57, mean 78.2, max 1524


In [9]:
print("labels per paper:")
print(df.n_labels.value_counts().sort_index().to_string())

print("\npapers per field:")
for c in label_cols:
    print(f"  {int(df[c].sum()):>5}  {c[4:]}")

counts = [int(df[c].sum()) for c in label_cols]
print(f"\nlargest/smallest class ratio: {max(counts) / min(counts):.2f}")


single = df[df.n_labels == 1]
print(f"\nIf restricted to single-label papers ({len(single)} papers):")
for c in label_cols:
    print(f"  {int(single[c].sum()):>5}  {c[4:]}")
s_counts = [int(single[c].sum()) for c in label_cols]
print(f"  ratio would become: {max(s_counts) / min(s_counts):.2f}")

labels per paper:
n_labels
1    5780
2    1488
3     924

papers per field:
   1000  Biotechnology
   1656  Chemistry
   1035  CS_AI
   1011  CS_InfoSystems
   1435  Engineering_EE
    988  Environmental
   1384  MaterialsScience
   1000  Mathematics
   1004  Physics_Applied
   1015  Psychology

largest/smallest class ratio: 1.68

If restricted to single-label papers (5780 papers):
    826  Biotechnology
    333  Chemistry
    543  CS_AI
    440  CS_InfoSystems
    502  Engineering_EE
    892  Environmental
    245  MaterialsScience
    917  Mathematics
     81  Physics_Applied
   1001  Psychology
  ratio would become: 12.36


In [10]:


vocabulary, row_idx, col_idx = {}, [], []

for i, refs in enumerate(df.ref_dois):
    for r in (x for x in refs.split("|") if x):
        j = vocabulary.setdefault(r, len(vocabulary))
        row_idx.append(i)
        col_idx.append(j)

M = csr_matrix((np.ones(len(row_idx)), (row_idx, col_idx)),
               shape=(len(df), len(vocabulary)))

print(f"unique cited references: {len(vocabulary):,}")
print(f"papers with no usable reference DOIs: "
      f"{int((np.asarray(M.sum(axis=1)).ravel() == 0).sum())}")

unique cited references: 367,569
papers with no usable reference DOIs: 6


In [11]:

times_cited = np.asarray(M.sum(axis=0)).ravel()
shared = times_cited >= 2
print(f"references shared by 2+ papers: {int(shared.sum()):,} "
      f"({100 * shared.mean():.1f}%)")


Mshared = M[:, shared]
C = (Mshared @ Mshared.T).tocsr()
C.setdiag(0)
C.eliminate_zeros()

partners = np.diff(C.indptr)
print(f"\npapers with at least one coupling partner: "
      f"{int((partners > 0).sum())} ({100 * (partners > 0).mean():.1f}%)")
print(f"papers with no partner at all: {int((partners == 0).sum())}")
print(f"median partners: {int(np.median(partners))}")

print("\nconnectivity at stricter thresholds:")
for t in (1, 2, 3, 5):
    strong = C.copy()
    strong.data = (strong.data >= t).astype(float)
    strong.eliminate_zeros()
    d = np.diff(strong.indptr)
    print(f"  >= {t} shared references: {int((d > 0).sum())} papers "
          f"({100 * (d > 0).mean():.1f}%)")

references shared by 2+ papers: 66,420 (18.1%)

papers with at least one coupling partner: 8124 (99.2%)
papers with no partner at all: 68
median partners: 62

connectivity at stricter thresholds:
  >= 1 shared references: 8124 papers (99.2%)
  >= 2 shared references: 8037 papers (98.1%)
  >= 3 shared references: 7896 papers (96.4%)
  >= 5 shared references: 7615 papers (93.0%)


In [12]:
from collections import Counter

all_cats = Counter()
for cats in df.wos_categories:
    for c in str(cats).split(";"):
        c = c.strip()
        if c:
            all_cats[c] += 1

print(f"distinct WoS categories across the dataset: {len(all_cats)}")
print(f"\nmost common 15:")
for cat, n in all_cats.most_common(15):
    print(f"  {n:>5}  {cat}")

print(f"\ncategories per paper: "
      f"median {df.wos_categories.apply(lambda s: len(str(s).split(';'))).median():.0f}")

distinct WoS categories across the dataset: 137

most common 15:
   1435  Engineering, Electrical & Electronic
   1352  Materials Science, Multidisciplinary
   1111  Chemistry, Multidisciplinary
   1063  Nanoscience & Nanotechnology
   1035  Computer Science, Artificial Intelligence
   1013  Chemistry, Physical
   1011  Computer Science, Information Systems
   1004  Physics, Applied
   1000  Biotechnology & Applied Microbiology
    988  Environmental Sciences
    894  Psychology, Multidisciplinary
    776  Physics, Condensed Matter
    700  Telecommunications
    556  Mathematics, Interdisciplinary Applications
    472  Energy & Fuels

categories per paper: median 2


In [13]:

from collections import Counter

all_cats = Counter()
for cats in df.wos_categories:
    for c in str(cats).split(";"):
        c = c.strip()
        if c:
            all_cats[c] += 1

print(f"distinct WoS categories in the corpus: {len(all_cats)}")
print(f"(of which {len([c for c in all_cats if all_cats[c] >= 100])} appear in 100+ papers)")


combos = df.labels.value_counts()
print(f"\nunique label combinations present: {len(combos)}")
print(f"theoretical maximum for 10 labels: {2**10 - 1}")
print(f"combinations with fewer than 10 papers: {(combos < 10).sum()}")
print(f"combinations with fewer than 30 papers: {(combos < 30).sum()}")

print(f"\nten most common combinations:")
print(combos.head(10).to_string())

print(f"\nten rarest combinations:")
print(combos.tail(10).to_string())

distinct WoS categories in the corpus: 137
(of which 43 appear in 100+ papers)

unique label combinations present: 36
theoretical maximum for 10 labels: 1023
combinations with fewer than 10 papers: 8
combinations with fewer than 30 papers: 15

ten most common combinations:
labels
Psychology                                    1001
Mathematics                                    917
Environmental                                  892
Chemistry|MaterialsScience|Physics_Applied     828
Biotechnology                                  826
CS_AI                                          543
Engineering_EE                                 502
CS_InfoSystems                                 440
CS_InfoSystems|Engineering_EE                  407
CS_AI|Engineering_EE                           380

ten rarest combinations:
labels
Engineering_EE|Physics_Applied                     11
Chemistry|CS_AI|Mathematics                        10
Biotechnology|MaterialsScience                      7
Chemistry|Math

In [14]:
n_dois = df.ref_dois.apply(lambda s: len([x for x in str(s).split("|") if x]))
print(f"reference DOIs per paper: median {n_dois.median():.0f}, "
      f"mean {n_dois.mean():.1f}, min {n_dois.min()}, max {n_dois.max()}, "
      f"papers with zero: {(n_dois == 0).sum()}")

reference DOIs per paper: median 49, mean 64.2, min 0, max 1301, papers with zero: 6
